In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import warnings
 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
 
warnings.filterwarnings("ignore")

In [3]:
df = pd.read_csv('dataset_clean_2.csv')

In [4]:
CSV_PATH     = "dataset_clean_2.csv"
TARGET_COL   = "popularity"
TEST_SIZE    = 0.2
RANDOM_STATE = 42
TOP_N_GENRE  = 10       # giữ top 10 genre phổ biến nhất theo popularity trung bình
OUTPUT_DIR   = "outputs"

In [6]:
# BƯỚC 1 — ĐỌC DỮ LIỆU
print("BƯỚC 1 — Đọc dữ liệu")
 
print(f"✅ Đọc xong: {df.shape[0]:,} hàng | {df.shape[1]} cột")
print(f"   Missing values : {df.isnull().sum().sum()}")
print(f"   Duplicates     : {df.duplicated().sum()}")

BƯỚC 1 — Đọc dữ liệu
✅ Đọc xong: 80,140 hàng | 15 cột
   Missing values : 0
   Duplicates     : 273


In [7]:
# BƯỚC 2 — DỌN DẸP CƠ BẢN (nếu còn sót)
# ═══════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("BƯỚC 2 — Dọn dẹp cơ bản")
print("=" * 55)
 
df = df.drop_duplicates()
df = df[df["time_signature"] > 0]   # time_signature=0 vô nghĩa về âm nhạc
df = df.reset_index(drop=True)
 
print(f"✅ Sau khi dọn: {df.shape[0]:,} hàng")
 


BƯỚC 2 — Dọn dẹp cơ bản
✅ Sau khi dọn: 79,862 hàng


In [8]:
# BƯỚC 3 — XỬ LÝ track_genre
# Dùng top 10 genre theo POPULARITY TRUNG BÌNH
# (đúng hơn là dùng theo số lượng bài — theo hướng mentor)
# ═══════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("BƯỚC 3 — Gộp genre (top 10 theo popularity trung bình)")
print("=" * 55)
 
# Tính popularity trung bình của từng genre
genre_avg_popularity = (
    df.groupby("track_genre")[TARGET_COL]
    .mean()
    .sort_values(ascending=False)
)


BƯỚC 3 — Gộp genre (top 10 theo popularity trung bình)


In [9]:
# Lấy top 10 genre có popularity trung bình cao nhất
top_genres = genre_avg_popularity.head(TOP_N_GENRE).index.tolist()
 
print(f"Top {TOP_N_GENRE} genres được giữ lại:")
for i, g in enumerate(top_genres, 1):
    avg = genre_avg_popularity[g]
    print(f"   {i:2d}. {g:<20s} (avg popularity: {avg:.1f})")

Top 10 genres được giữ lại:
    1. k-pop                (avg popularity: 59.7)
    2. pop                  (avg popularity: 59.3)
    3. pop-film             (avg popularity: 59.2)
    4. electro              (avg popularity: 58.3)
    5. metal                (avg popularity: 57.9)
    6. latino               (avg popularity: 57.5)
    7. soul                 (avg popularity: 57.2)
    8. edm                  (avg popularity: 57.2)
    9. house                (avg popularity: 56.6)
   10. hip-hop              (avg popularity: 56.0)


In [10]:
# Gộp phần còn lại thành "Others"
df["genre_grouped"] = df["track_genre"].apply(
    lambda x: x if x in top_genres else "Others"
)
 
print(f"\n   Phân phối sau khi gộp:")
print(df["genre_grouped"].value_counts().to_string())


   Phân phối sau khi gộp:
genre_grouped
Others      75462
k-pop         912
pop-film      809
hip-hop       638
edm           491
latino        357
electro       350
pop           292
metal         225
soul          207
house         119


In [11]:
# Bỏ cột gốc track_genre (đã có genre_grouped thay thế)
df = df.drop(columns=["track_genre"])

In [12]:
# BƯỚC 4 — TÁCH X VÀ y
# ═══════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("BƯỚC 4 — Tách X (features) và y (target)")
print("=" * 55)
 
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]


BƯỚC 4 — Tách X (features) và y (target)


In [13]:
# Phân loại cột
cat_cols     = ["genre_grouped"]
numeric_cols = [c for c in X.columns if c not in cat_cols]
 
print(f"   X shape       : {X.shape}")
print(f"   y shape       : {y.shape}")
print(f"   Numeric cols  : {numeric_cols}")
print(f"   Categorical   : {cat_cols}")

   X shape       : (79862, 14)
   y shape       : (79862,)
   Numeric cols  : ['danceability', 'energy', 'valence', 'acousticness', 'speechiness', 'instrumentalness', 'liveness', 'tempo', 'duration_ms', 'key', 'mode', 'time_signature', 'explicit']
   Categorical   : ['genre_grouped']


In [14]:
# BƯỚC 5 — CHIA TRAIN / TEST
# *** Phải chia TRƯỚC khi encode — tránh data leakage ***
# ═══════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("BƯỚC 5 — Chia train / test set")
print("=" * 55)
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)
 
print(f"✅ Train : {X_train.shape[0]:,} hàng ({(1-TEST_SIZE)*100:.0f}%)")
print(f"   Test  : {X_test.shape[0]:,} hàng ({TEST_SIZE*100:.0f}%)")
print()
print("⚠️  Từ đây KHÔNG được dùng X_test/y_test cho đến bước đánh giá model cuối!")
 


BƯỚC 5 — Chia train / test set
✅ Train : 63,889 hàng (80%)
   Test  : 15,973 hàng (20%)

⚠️  Từ đây KHÔNG được dùng X_test/y_test cho đến bước đánh giá model cuối!


In [15]:
# BƯỚC 6 — XÂY DỰNG PREPROCESSOR PIPELINE
# Fit TRÊN TRAIN SET → transform cả train lẫn test
# (đúng quy trình, tránh data leakage)
# ═══════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("BƯỚC 6 — Build & fit preprocessor (chỉ fit trên train)")
print("=" * 55)
 
# Nhánh xử lý cột số: điền missing → chuẩn hóa
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])
 
# Nhánh xử lý cột categorical: điền missing → One-Hot Encode
# drop="first" để tránh dummy variable trap (multicollinearity)
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",   # genre lạ ở test set → tất cả = 0, không lỗi
        sparse_output=False,
        drop="first",
    )),
])
 
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline,     numeric_cols),
    ("cat", categorical_pipeline, cat_cols),
])
 
# Fit TRÊN TRAIN rồi transform cả hai
X_train_encoded = preprocessor.fit_transform(X_train)   # fit + transform
X_test_encoded  = preprocessor.transform(X_test)        # chỉ transform, không fit lại
 
# Lấy tên cột sau khi encode (để DataFrame dễ đọc)
ohe_feature_names = (
    preprocessor
    .named_transformers_["cat"]
    .named_steps["encoder"]
    .get_feature_names_out(cat_cols)
    .tolist()
)
all_feature_names = numeric_cols + ohe_feature_names
 
# Chuyển về DataFrame để lưu CSV
X_train_encoded = pd.DataFrame(X_train_encoded, columns=all_feature_names)
X_test_encoded  = pd.DataFrame(X_test_encoded,  columns=all_feature_names)
 
print(f"✅ Encode xong!")
print(f"   Số cột trước encode : {len(numeric_cols) + len(cat_cols)}")
print(f"   Số cột sau encode   : {X_train_encoded.shape[1]}")
print(f"   Các cột genre mới   : {ohe_feature_names}")
 
 


BƯỚC 6 — Build & fit preprocessor (chỉ fit trên train)
✅ Encode xong!
   Số cột trước encode : 14
   Số cột sau encode   : 23
   Các cột genre mới   : ['genre_grouped_edm', 'genre_grouped_electro', 'genre_grouped_hip-hop', 'genre_grouped_house', 'genre_grouped_k-pop', 'genre_grouped_latino', 'genre_grouped_metal', 'genre_grouped_pop', 'genre_grouped_pop-film', 'genre_grouped_soul']


In [16]:
# BƯỚC 7 — LƯU 4 FILE CSV + PREPROCESSOR
# → Người làm modeling chỉ cần load 4 file này
# ═══════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("BƯỚC 7 — Lưu file output")
print("=" * 55)
 
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
# Lưu 4 file csv
X_train_encoded.to_csv(f"{OUTPUT_DIR}/X_train.csv", index=False)
X_test_encoded.to_csv( f"{OUTPUT_DIR}/X_test.csv",  index=False)
y_train.to_csv(        f"{OUTPUT_DIR}/y_train.csv",  index=False)
y_test.to_csv(         f"{OUTPUT_DIR}/y_test.csv",   index=False)
 
# Lưu preprocessor (để dùng lại khi predict dữ liệu mới sau này)
with open(f"{OUTPUT_DIR}/preprocessor.pkl", "wb") as f:
    pickle.dump(preprocessor, f)
 
# Lưu danh sách feature names (người làm modeling cần biết)
with open(f"{OUTPUT_DIR}/feature_names.pkl", "wb") as f:
    pickle.dump(all_feature_names, f)
 
print(f"✅ Đã lưu vào thư mục '{OUTPUT_DIR}/':")
print(f"   X_train.csv  — {X_train_encoded.shape[0]:,} hàng × {X_train_encoded.shape[1]} cột")
print(f"   X_test.csv   — {X_test_encoded.shape[0]:,} hàng × {X_test_encoded.shape[1]} cột")
print(f"   y_train.csv  — {len(y_train):,} hàng")
print(f"   y_test.csv   — {len(y_test):,} hàng")
print(f"   preprocessor.pkl  — dùng để predict dữ liệu mới")
print(f"   feature_names.pkl — danh sách tên cột")


BƯỚC 7 — Lưu file output
✅ Đã lưu vào thư mục 'outputs/':
   X_train.csv  — 63,889 hàng × 23 cột
   X_test.csv   — 15,973 hàng × 23 cột
   y_train.csv  — 63,889 hàng
   y_test.csv   — 15,973 hàng
   preprocessor.pkl  — dùng để predict dữ liệu mới
   feature_names.pkl — danh sách tên cột


In [17]:
# KIỂM TRA NHANH — load lại và verify
# ═══════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("KIỂM TRA — Load lại file và verify")
print("=" * 55)
 
X_train_check = pd.read_csv(f"{OUTPUT_DIR}/X_train.csv")
X_test_check  = pd.read_csv(f"{OUTPUT_DIR}/X_test.csv")
y_train_check = pd.read_csv(f"{OUTPUT_DIR}/y_train.csv")
y_test_check  = pd.read_csv(f"{OUTPUT_DIR}/y_test.csv")
 
print(f"   X_train : {X_train_check.shape} ✅")
print(f"   X_test  : {X_test_check.shape}  ✅")
print(f"   y_train : {y_train_check.shape} ✅")
print(f"   y_test  : {y_test_check.shape}  ✅")
print(f"\n   Missing trong X_train: {X_train_check.isnull().sum().sum()}")
print(f"   Missing trong X_test : {X_test_check.isnull().sum().sum()}")
 
print("\n" + "=" * 55)
print("🎉 PREPROCESSING XONG!")
print("   Gửi thư mục 'outputs/' cho người làm modeling.")
print("   Họ chỉ cần load 4 file CSV là train model được ngay.")
print("=" * 55)


KIỂM TRA — Load lại file và verify
   X_train : (63889, 23) ✅
   X_test  : (15973, 23)  ✅
   y_train : (63889, 1) ✅
   y_test  : (15973, 1)  ✅

   Missing trong X_train: 0
   Missing trong X_test : 0

🎉 PREPROCESSING XONG!
   Gửi thư mục 'outputs/' cho người làm modeling.
   Họ chỉ cần load 4 file CSV là train model được ngay.
